[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/01.%20The%20Single%20Crane%20Lift%20Sequence%20Problem/P1-Tier-4_executed.ipynb)

---


# 1. The Single Crane Lift Sequence Problem

## Tier 4 — Reinforcement learning with a linear DQN-style agent

This notebook models a crane's lift sequence as a sequential decision problem. At each step, the agent observes which containers remain, the crane's location, and its current lift mode. It then chooses a feasible single lift or dual lift, receives a reward based on the time spent, and observes the resulting state.

The implementation uses a **linear action-value model** rather than a neural-network library. It retains the main components of a DQN-style learning pipeline: epsilon-greedy action selection, experience replay, and temporal-difference (TD) updates.

### Learning goals

- Describe the lift-sequencing task as a Markov decision process (MDP): state, action, reward, and transition.
- Explain how a negative time-based reward turns time minimization into reward maximization.
- Follow how a linear model estimates action values and how replay updates those estimates.
- Evaluate the learned greedy policy and interpret its schedule and total time.

### Inputs, outputs, and scope

The input is a fixed six-container instance, a 2-by-3 grid layout, lift-time parameters, a dual-lift weight limit, and a reproducible random seed. The notebook trains an agent on this instance, plots training diagnostics, and reports the schedule selected by its greedy policy.

The example is a small simulation for learning the method. It does not model vessel arrivals, weather, equipment failures, uncertain processing times, or interactions among multiple cranes. Training on one fixed instance does not establish that a policy will generalize to other instances.

In [ ]:
# Check that the required packages are available without installing them.
# For a shared teaching environment, dependencies can be preinstalled in its image.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install numpy, pandas, and matplotlib in your environment."
    ) from e

rng = np.random.default_rng(0)
print("Dependencies imported successfully.")


In [ ]:
# ----------------------------
# Imports and problem setup
# ----------------------------
# The fixed action list lets the agent compare the same action indices in every state.

from dataclasses import dataclass
from itertools import combinations
from collections import deque
from typing import List, Tuple, Dict, Any, Set


# ----------------------------
# Container representation
# ----------------------------
# Container positions are labels on a 2-by-3 grid; coordinates are defined below.
@dataclass(frozen=True)
class Container:
    id: int
    pos: int
    weight: float


# Concrete six-container instance
containers = [
    Container(1, 1, 8.5),
    Container(2, 2, 12.3),
    Container(3, 3, 15.7),
    Container(4, 4, 9.8),
    Container(5, 5, 11.2),
    Container(6, 6, 14.1),
]

id_to_container = {c.id: c for c in containers}


# ----------------------------
# Parameters (minutes, weight units, and grid units)
# ----------------------------
HANDLE_SINGLE = 2.5
HANDLE_DUAL = 3.8
SETUP_TIME = 1.2
DUAL_WEIGHT_LIMIT = 20.0
TRAVEL_PER_UNIT = 0.8

START_POS = (0.0, 0.0)
START_MODE = 0  # 0 = single mode, 1 = dual mode


# ----------------------------
# Grid layout and adjacency
# ----------------------------
# Serpentine numbering:
# Row 0: 1 2 3
# Row 1: 6 5 4
# Crane travel uses Manhattan distance on this same grid.
pos_to_rc = {
    1: (0, 0),
    2: (0, 1),
    3: (0, 2),
    4: (1, 2),
    5: (1, 1),
    6: (1, 0),
}


def are_adjacent(p1: int, p2: int) -> bool:
    """Return whether two labeled positions share a grid edge."""
    r1, c1 = pos_to_rc[p1]
    r2, c2 = pos_to_rc[p2]
    return abs(r1 - r2) + abs(c1 - c2) == 1


def travel_time(
    pos_from: Tuple[float, float],
    pos_to: Tuple[float, float],
) -> float:
    """Compute crane travel time from Manhattan distance on the grid."""
    distance = abs(pos_from[0] - pos_to[0]) + abs(pos_from[1] - pos_to[1])
    return TRAVEL_PER_UNIT * distance


def dual_center(i: int, j: int) -> Tuple[float, float]:
    """Return the coordinate midway between the two dual-lift positions."""
    row_i, col_i = pos_to_rc[id_to_container[i].pos]
    row_j, col_j = pos_to_rc[id_to_container[j].pos]
    return (row_i + row_j) / 2.0, (col_i + col_j) / 2.0


def dual_feasible(i: int, j: int) -> bool:
    """A dual lift requires adjacent containers within the weight limit."""
    return are_adjacent(id_to_container[i].pos, id_to_container[j].pos) and (
        id_to_container[i].weight + id_to_container[j].weight
    ) <= DUAL_WEIGHT_LIMIT


# ----------------------------
# Fixed action space
# ----------------------------
# Each action is either ('S', container_id) or ('D', container_id_1, container_id_2).
ALL_SINGLE = [("S", c.id) for c in containers]
ALL_DUAL = [
    ("D", i, j)
    for i, j in combinations([c.id for c in containers], 2)
    if dual_feasible(i, j)
]
ACTIONS = ALL_SINGLE + ALL_DUAL

# State features: six remaining-container indicators, two position coordinates,
# and one lift-mode indicator.
STATE_SIZE = 6 + 2 + 1
ACTION_SIZE = len(ACTIONS)

STATE_SIZE, ACTION_SIZE, ACTIONS


## Checking dual-action availability

The main instance uses the specified dual-lift weight limit. If no adjacent pair meets that limit, the agent cannot choose a dual lift in this instance, even though the environment supports dual actions in general. The what-if table below counts feasible adjacent pairs at several limits; it is a sensitivity check, not a change to the main instance.

A dual pair must satisfy both requirements:

1. Its container positions share a grid edge.
2. The sum of its container weights does not exceed the selected limit.

In [ ]:
# Count the dual actions available under the main instance parameters.
num_dual_actions = int(len(ALL_DUAL))
print("Dual actions in ACTIONS:", num_dual_actions)
print("Dual actions list:", ALL_DUAL)


def dual_pairs_for_limit(limit: float) -> List[Tuple[int, int]]:
    """List adjacent container pairs whose combined weight meets a limit."""
    pairs = []
    ids = [c.id for c in containers]
    for i, j in combinations(ids, 2):
        if not are_adjacent(id_to_container[i].pos, id_to_container[j].pos):
            continue
        if id_to_container[i].weight + id_to_container[j].weight <= float(limit):
            pairs.append((i, j))
    return pairs


limits_to_try = [20.0, 22.0, 24.0, 26.0, 30.0]
rows = []
for limit in limits_to_try:
    pairs = dual_pairs_for_limit(limit)
    rows.append(
        {
            "dual_weight_limit": float(limit),
            "num_dual_pairs": int(len(pairs)),
            "dual_pairs": pairs,
        }
    )

pd.DataFrame(rows)


## Environment dynamics

### State and action

A state records which containers remain, the crane's grid coordinate, and its current lift mode. An action is one element of the fixed `ACTIONS` list. A single action removes one remaining container; a dual action removes two remaining containers at once.

### Time cost and reward

Each valid action incurs travel time, handling time, and—if the lift mode changes—setup time. The reward is the negative of that action's time cost:

\[
r_t = -c(s_t,a_t),
\]

where \(c(s_t,a_t)\) is the step time. Thus, maximizing accumulated reward corresponds to minimizing accumulated time when rewards are summed without discounting. The learning agent uses a discount factor below one, so its TD learning target is a discounted proxy for total time.

### Invalid actions and terminal states

An invalid action, such as lifting a container that has already been removed, receives a large penalty and leaves the physical state unchanged. The policy masks invalid actions during action selection, so this penalty is a safeguard rather than the normal route through the episode. An episode ends when no containers remain.

In [ ]:
# ----------------------------
# Environment functions
# ----------------------------
# The state vector makes the discrete remaining set and continuous crane location
# available to the linear action-value model.


def encode_state(
    remaining: Set[int],
    pos: Tuple[float, float],
    mode: int,
) -> np.ndarray:
    """Convert the environment state into normalized numeric features."""
    mask = np.array(
        [1.0 if i in remaining else 0.0 for i in range(1, 7)],
        dtype=np.float32,
    )

    # Grid rows range from 0 to 1 and columns from 0 to 2.
    pos_norm = np.array([pos[0] / 1.0, pos[1] / 2.0], dtype=np.float32)
    mode_arr = np.array([float(mode)], dtype=np.float32)
    return np.concatenate([mask, pos_norm, mode_arr])


def valid_action_mask(remaining: Set[int]) -> np.ndarray:
    """Mark actions that can be performed with the remaining containers."""
    mask = np.zeros((ACTION_SIZE,), dtype=np.float32)

    for k, act in enumerate(ACTIONS):
        if act[0] == "S":
            if act[1] in remaining:
                mask[k] = 1.0
        elif act[1] in remaining and act[2] in remaining:
            mask[k] = 1.0

    return mask


def step_env(
    remaining: Set[int],
    pos: Tuple[float, float],
    mode: int,
    action_idx: int,
):
    """Apply an action and return the next state, reward, terminal flag, and info."""
    act = ACTIONS[action_idx]

    def cost(lift_mode: int, target_pos: Tuple[float, float]) -> float:
        # Setup is needed only when changing between single and dual lift modes.
        setup = SETUP_TIME if mode != lift_mode else 0.0
        handle = HANDLE_DUAL if lift_mode == 1 else HANDLE_SINGLE
        return travel_time(pos, target_pos) + setup + handle

    info: Dict[str, Any] = {"action": act}

    if act[0] == "S":
        cid = act[1]
        if cid not in remaining:
            return remaining, pos, mode, -50.0, False, {**info, "invalid": True}

        row, col = pos_to_rc[id_to_container[cid].pos]
        target = (float(row), float(col))
        step_time = cost(0, target)

        new_remaining = set(remaining)
        new_remaining.remove(cid)
        reward = -step_time
        done = len(new_remaining) == 0
        return new_remaining, target, 0, reward, done, {**info, "step_time": step_time}

    if act[0] == "D":
        i, j = act[1], act[2]
        if i not in remaining or j not in remaining or not dual_feasible(i, j):
            return remaining, pos, mode, -50.0, False, {**info, "invalid": True}

        target = dual_center(i, j)
        step_time = cost(1, target)

        new_remaining = set(remaining)
        new_remaining.remove(i)
        new_remaining.remove(j)
        reward = -step_time
        done = len(new_remaining) == 0
        return new_remaining, target, 1, reward, done, {**info, "step_time": step_time}

    raise ValueError("Unknown action")


# A one-step check connects the feature representation to an environment transition.
rem0 = set(range(1, 7))
s0 = encode_state(rem0, START_POS, START_MODE)
rem1, pos1, mode1, r1, done1, info1 = step_env(rem0, START_POS, START_MODE, 0)

assert s0.shape == (STATE_SIZE,)
assert len(rem1) == len(rem0) - 1
assert np.isclose(r1, -info1["step_time"])
(s0.shape, rem1, pos1, mode1, r1, done1, info1)


## Linear DQN-style agent

### Action values

The agent estimates how useful each action is in a state. This estimate is called an action value, or Q-value. For action \(a\), the linear approximation is

\[
Q(s,a) = w_a^\mathsf{T}s + b_a,
\]

where \(s\) is the state-feature vector, \(w_a\) is the feature-weight vector for action \(a\), and \(b_a\) is its bias. The implementation stores all action weights in `W` and all biases in `b`.

A linear model is less expressive than a neural network, but its predictions are easy to inspect and it keeps the learning mechanics visible. The method here uses epsilon-greedy exploration and a replay buffer. It is DQN-style rather than a full neural-network DQN: there is no target network, and the linear weights are updated directly.

### TD learning and replay

The agent compares its current estimate with a one-step TD target. For a nonterminal transition, the target uses the best estimated value among valid next actions:

\[
y_t = r_t + \gamma \max_{a'\in\mathcal{A}(s_{t+1})} Q(s_{t+1},a'),
\]

and for a terminal transition, \(y_t=r_t\). The TD error is \(\delta_t=y_t-Q(s_t,a_t)\). For the selected action, the linear parameters are updated by

\[
w_{a_t} \leftarrow w_{a_t} + \alpha\,\delta_t s_t,
\qquad
b_{a_t} \leftarrow b_{a_t} + \alpha\,\delta_t.
\]

Here \(\gamma\) is the discount factor and \(\alpha\) is the learning rate. Experience replay stores transitions and samples them in batches, reducing the dependence of each update on the immediately preceding transition. In the code, `replay` calculates the TD target and applies these updates.

In [ ]:
# ----------------------------
# Linear DQN-style agent
# ----------------------------


class LinearDQN:
    def __init__(
        self,
        state_size: int,
        action_size: int,
        lr: float = 0.05,
        gamma: float = 0.98,
    ):
        self.state_size = state_size
        self.action_size = action_size
        self.lr = lr
        self.gamma = gamma

        # Each action has one weight per feature and one bias.
        self.W = rng.normal(0, 0.1, size=(action_size, state_size)).astype(np.float32)
        self.b = np.zeros((action_size,), dtype=np.float32)

        # Stored transitions support updates from a shuffled sample of experience.
        self.memory = deque(maxlen=20000)

    def q_values(self, s: np.ndarray) -> np.ndarray:
        """Compute estimated values for every action in one state."""
        return (self.W @ s) + self.b

    def act(self, s: np.ndarray, valid_mask: np.ndarray, eps: float) -> int:
        """Choose a random valid action or the highest-valued valid action."""
        valid_indices = np.where(valid_mask > 0)[0]
        if len(valid_indices) == 0:
            raise ValueError("No valid action is available in this state.")

        if rng.random() < eps:
            return int(rng.choice(valid_indices))

        q = self.q_values(s)
        q = np.where(valid_mask > 0, q, -1e9)
        return int(np.argmax(q))

    def remember(self, s, a, r, s2, done, valid_mask2):
        """Store one transition for later replay updates."""
        self.memory.append((s, a, r, s2, done, valid_mask2))

    def replay(self, batch_size: int = 64) -> float:
        """Apply linear TD updates to a random batch and return mean squared error."""
        if len(self.memory) < batch_size:
            return 0.0

        batch = rng.choice(len(self.memory), size=batch_size, replace=False)
        losses = []

        for idx in batch:
            s, a, r, s2, done, valid_mask2 = self.memory[idx]
            q = self.q_values(s)

            target = float(r)
            if not done:
                q2 = self.q_values(s2)
                q2 = np.where(valid_mask2 > 0, q2, -1e9)
                target = float(r) + self.gamma * float(np.max(q2))

            err = target - float(q[a])

            # The gradient of a linear Q-value with respect to its action weights is s.
            self.W[a] += self.lr * err * s
            self.b[a] += self.lr * err
            losses.append(err * err)

        return float(np.mean(losses))


# One agent is trained and evaluated on the fixed instance.
agent = LinearDQN(STATE_SIZE, ACTION_SIZE)


In [ ]:
# ----------------------------
# Training loop
# ----------------------------
# An episode starts with every container present and ends after all are lifted.


def run_episode(agent: LinearDQN, eps: float, train: bool = True, max_steps: int = 50):
    """Run one episode and optionally update the supplied agent from replay."""
    remaining = set(range(1, 7))
    pos = START_POS
    mode = START_MODE

    total_reward = 0.0
    total_time = 0.0
    losses = []

    for _ in range(max_steps):
        s = encode_state(remaining, pos, mode)
        vmask = valid_action_mask(remaining)
        a = agent.act(s, vmask, eps=eps)

        remaining2, pos2, mode2, reward, done, info = step_env(
            remaining, pos, mode, a
        )
        total_reward += float(reward)
        if "step_time" in info:
            total_time += float(info["step_time"])

        s2 = encode_state(remaining2, pos2, mode2)
        vmask2 = valid_action_mask(remaining2)

        if train:
            agent.remember(s, a, reward, s2, done, vmask2)
            losses.append(agent.replay(batch_size=64))

        remaining, pos, mode = remaining2, pos2, mode2
        if done:
            break

    mean_loss = float(np.mean(losses) if losses else 0.0)
    return total_time, total_reward, mean_loss


# ----------------------------
# Run training
# ----------------------------
episodes = 450

eps0 = 1.0
eps_min = 0.05
decay = 0.993

ep_times, ep_rewards, ep_losses = [], [], []
eps = eps0

for ep in range(episodes):
    episode_time, episode_reward, loss = run_episode(
        agent, eps=eps, train=True
    )
    ep_times.append(episode_time)
    ep_rewards.append(episode_reward)
    ep_losses.append(loss)
    eps = max(eps_min, eps * decay)

print("Training finished.")
print(f"Final epsilon: {eps:.3f}")
print(f"Last-20 average time: {np.mean(ep_times[-20:]):.3f} min")


## Training curves

The plots show three diagnostics recorded during training:

- **Episode total time:** the sum of the valid step times in an episode. A downward trend is desirable, but the curve can fluctuate because action selection includes exploration and learning updates are stochastic.
- **Episode reward:** the sum of step rewards. Since each valid reward is negative time, a higher (less negative) return corresponds to a shorter episode time when the undiscounted rewards are summed.
- **Mean TD loss:** the average squared TD error recorded during an episode. It can be noisy; it measures prediction error, not schedule quality directly.

The discount factor used for TD targets means the learning update optimizes a discounted return proxy. The reported episode time remains the direct operational measure for the schedule.

In [ ]:
# Plot the recorded time, reward, and TD-loss diagnostics.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

axes[0].plot(ep_times, lw=1)
axes[0].set_title("Episode total time")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Time (min)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_rewards, lw=1)
axes[1].set_title("Episode reward")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Reward (negative minutes)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep_losses, lw=1)
axes[2].set_title("Mean TD loss per episode")
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Mean squared TD error")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Greedy policy evaluation

Evaluation sets \(\epsilon=0\), so the agent chooses the highest-valued valid action at each step rather than exploring randomly. The resulting table records the chosen action, its duration, and the containers remaining after that action.

This is one rollout of the learned policy, not a proof of optimality. In particular, the main instance's dual-weight limit determines whether any dual actions are available to the agent.

In [ ]:
# ----------------------------
# Evaluate the greedy policy
# ----------------------------


def evaluate_greedy(agent: LinearDQN):
    """Run one deterministic greedy rollout and return its time and step table."""
    remaining = set(range(1, 7))
    pos = START_POS
    mode = START_MODE
    log = []
    total_time = 0.0

    for _ in range(50):
        s = encode_state(remaining, pos, mode)
        vmask = valid_action_mask(remaining)
        action_idx = agent.act(s, vmask, eps=0.0)

        remaining2, pos2, mode2, reward, done, info = step_env(
            remaining, pos, mode, action_idx
        )
        log.append(
            {
                "action": info["action"],
                "step_time": info.get("step_time", None),
                "remaining": sorted(remaining2),
            }
        )

        if "step_time" in info:
            total_time += float(info["step_time"])

        remaining, pos, mode = remaining2, pos2, mode2
        if done:
            break

    return total_time, pd.DataFrame(log)


greedy_time, greedy_df = evaluate_greedy(agent)
assert np.isclose(
    greedy_time,
    greedy_df["step_time"].dropna().astype(float).sum(),
)
print(f"Greedy policy total time: {greedy_time:.3f} min")
greedy_df


## Operational visualizations

The following plots summarize the greedy evaluation rollout. Step durations show the time charged to each decision, cumulative time shows how those costs build up, and action counts distinguish single from dual lifts. The action-type colors are identified by the category labels on the third plot.

In [ ]:
# Summarize the greedy rollout without changing its recorded decisions.
vis = greedy_df.copy()
vis["action_type"] = vis["action"].apply(lambda action: action[0])
vis["action_label"] = vis["action_type"].map({"S": "Single", "D": "Dual"})
vis.index = np.arange(1, len(vis) + 1)
vis["step"] = vis.index.astype(int)

# Invalid steps have no duration; only valid actions contribute to time summaries.
vis_valid = vis.dropna(subset=["step_time"]).copy()
vis_valid["step_time"] = vis_valid["step_time"].astype(float)
vis_valid["cum_time"] = vis_valid["step_time"].cumsum()

color_map = {"Single": "#2E90FA", "Dual": "#12B76A"}
colors = [color_map[label] for label in vis_valid["action_label"].tolist()]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))

axes[0].bar(
    vis_valid["step"],
    vis_valid["step_time"],
    color=colors,
    edgecolor="#101828",
    alpha=0.85,
)
axes[0].set_title("Greedy evaluation: step durations")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Time (min)")
axes[0].grid(True, axis="y", alpha=0.25)

axes[1].plot(
    vis_valid["step"],
    vis_valid["cum_time"],
    marker="o",
    lw=2,
    color="#344054",
)
axes[1].set_title("Greedy evaluation: cumulative time")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Time (min)")
axes[1].grid(True, alpha=0.25)

counts = (
    vis_valid["action_label"]
    .value_counts()
    .reindex(["Single", "Dual"])
    .fillna(0)
)
axes[2].bar(
    counts.index,
    counts.values,
    color=[color_map["Single"], color_map["Dual"]],
    edgecolor="#101828",
    alpha=0.85,
)
axes[2].set_title("Greedy evaluation: action counts")
axes[2].set_xlabel("Action type")
axes[2].set_ylabel("Number of actions")
axes[2].grid(True, axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


## Role of reinforcement learning in the problem-solving toolkit

Dynamic programming, greedy scheduling, genetic algorithms, and reinforcement learning address scheduling decisions in different ways. Dynamic programming can provide optimal solutions for suitably small state spaces; greedy rules construct schedules directly; genetic algorithms search among candidate schedules; and reinforcement learning estimates a policy through interaction with an environment.

A learned policy can be useful when many related decisions must be made repeatedly and representative simulation experience is available. However, policy quality depends on the state representation, reward design, training data, and evaluation procedure. Learning can be unstable, and a policy trained on one fixed instance should not be assumed to generalize. Unlike an exact optimizer for a specified instance, this agent does not provide an optimality guarantee.

This notebook demonstrates the sequential-decision formulation and a compact replay-based TD learner. Its fixed instance is suitable for inspecting the mechanics; broader claims about performance require evaluation on additional instances and operating conditions.